# 11 — 후보 B 전체 OOF/Test 생성 (팀장 승인)

- 후보 A는 보류. 후보 B는 **개발 CV 개선 + 홀드아웃 하락 0.02 이내** 기준으로 통과.
- 후보 B의 XGB 위치 모델을 CPU에서 전체 5-fold OOF 학습하고 전체 train 재학습으로 test 예측 생성.
- 기존 예측 파일과 합쳐 B 앙상블 OOF/Test 및 메타데이터 저장. 제출 CSV는 생성하지 않음.
- PR은 결과 검증 후 다음 날 진행.


In [1]:
from pathlib import Path
import sys, json, warnings
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.base import clone
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import f1_score
from xgboost import XGBClassifier

start = Path.cwd().resolve()
ROOT = next((p for p in (start, *start.parents) if (p/'src'/'config.py').is_file() and (p/'scripts'/'fixed_holdout_manifest.py').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('src/config.py가 있는 프로젝트 폴더에서 실행하세요.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# 노트북의 현재 작업 폴더와 무관하게 프로젝트 스크립트 로딩
import importlib.util
manifest_path = ROOT / 'scripts' / 'fixed_holdout_manifest.py'
spec = importlib.util.spec_from_file_location('fixed_holdout_manifest', manifest_path)
holdout_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(holdout_module)
assert holdout_module.ROOT.resolve() == ROOT.resolve(), '스크립트의 ROOT 경로를 확인하세요.'
from src.config import DATA_DIR, SEED, set_seed
from src.data_loader import load_data, split_xy, encode_target
from src.preprocessing import binarize_mutations
from src.validation import get_folds, run_cv
from src.evaluation import macro_f1, save_predictions, log_result
from src.features import mutation_count, mutation_type_counts, amino_acid_pair_counts, gene_truncation_flags

set_seed()
RUN_FULL_CV = True    # 팀장 승인 후 True: 전체 OOF/Test 생성 (기존 정상 파일 재사용)
SAVE_ARTIFACTS = True # True: 3개 모델 + 앙상블 예측 파일 저장
SAVE_SUBMISSION = False # 팀 규칙: 사용하지 않음
print('ROOT', ROOT, '| XGBoost', __import__('xgboost').__version__)
print('홀드아웃 스크립트:', manifest_path)

ROOT C:\Users\user\Desktop\AI_Projects\ah07-cancer-subtype | XGBoost 3.4.1
홀드아웃 스크립트: C:\Users\user\Desktop\AI_Projects\ah07-cancer-subtype\scripts\fixed_holdout_manifest.py


In [2]:
# 최종 제출 노트북과 일치하도록 XGBoost는 CPU로 고정합니다.
DEVICE = 'cpu'
print('XGBoost device:', DEVICE)


XGBoost device: cpu


In [3]:
train, test = load_data(DATA_DIR)
X_raw, y_raw = split_xy(train)
y, le = encode_target(y_raw)
X_test_raw = test[X_raw.columns].copy()
assert X_raw.columns.equals(X_test_raw.columns)
assert X_raw.index.equals(pd.RangeIndex(len(X_raw)))
assert len(le.classes_) == 26
# 공통 홀드아웃 먼저 읽기: CV는 개발 데이터에서만 실행합니다.
dev_idx, hold_idx = holdout_module.verify()
assert len(dev_idx) + len(hold_idx) == len(y)
assert not np.intersect1d(dev_idx, hold_idx).size
X_dev_raw = X_raw.iloc[dev_idx].reset_index(drop=True)
y_dev = np.asarray(y)[dev_idx]
folds = get_folds(y_dev, X=X_dev_raw, classes=le.classes_)  # 팀 함수 변경 금지
print('train', X_raw.shape, 'test', X_test_raw.shape, 'classes', len(le.classes_))

def make_E(X):
    # 시윤님 02_features E와 동일한 피처 순서/변환
    parts = [binarize_mutations(X), np.log1p(mutation_count(X)),
             np.log1p(mutation_type_counts(X)), np.log1p(amino_acid_pair_counts(X)),
             gene_truncation_flags(X)]
    out = pd.concat(parts, axis=1).astype(np.float32)
    assert not out.columns.has_duplicates and out.index.equals(X.index)
    assert np.isfinite(out.to_numpy()).all()
    return out

X_E = make_E(X_raw)
X_test_E = make_E(X_test_raw)
assert X_E.columns.equals(X_test_E.columns)
assert X_E.shape[1] == 9174, f'E 피처 개수 불일치: {X_E.shape[1]}'
print('E', X_E.shape, X_test_E.shape)
X_dev_E = X_E.iloc[dev_idx].reset_index(drop=True)


VERIFIED: dev=4960, holdout=1241; same IDs and row signatures
train (6201, 4384) test (2546, 4384) classes 26
E (6201, 9174) (2546, 9174)


## 1. 기존 3번째 모델: E + Position-bin LR (그대로 유지)

In [4]:
# 시윤님 03_hotspot_bins.ipynb의 E+Position-bin 함수 재현 (핫스팟 미사용)
import re
from collections import Counter
from scipy.sparse import hstack
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer

def make_features_E_sparse(X):
    return csr_matrix(make_E(X).to_numpy(dtype=np.float32))

def extract_position(variant):
    match = re.search(r"\d+", variant)
    return int(match.group()) if match else None
# Position-bin 후보 생성 함수

from collections import defaultdict

def extract_position_bins(X, bin_size=50):
    rows = []
    cols = []
    gene_bins = set()

    for row_idx, (_, row) in enumerate(X.iterrows()):
        patient_bins = set()

        for gene, value in row.items():
            if gene == "ID":
                continue

            if pd.isna(value) or str(value) == "WT":
                continue

            for variant in str(value).split():
                pos = extract_position(variant)

                if pos is not None:
                    feature = f"{gene}_bin_{pos // bin_size}"
                    patient_bins.add(feature)

        for feature in patient_bins:
            rows.append(row_idx)
            cols.append(feature)
            gene_bins.add(feature)

    return rows, cols, gene_bins


# Fold별 Position-bin 변환기 구현

from sklearn.base import BaseEstimator, TransformerMixin
from scipy.sparse import csr_matrix
from collections import Counter

class PositionBinTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, bin_size=50, min_patients=5):
        self.bin_size = bin_size
        self.min_patients = min_patients

    def fit(self, X, y=None):
        _, cols, _ = extract_position_bins(
            X, bin_size=self.bin_size
        )

        counts = Counter(cols)

        self.selected_bins_ = sorted(
            feature
            for feature, count in counts.items()
            if count >= self.min_patients
        )

        self.bin_to_idx_ = {
            feature: idx
            for idx, feature in enumerate(self.selected_bins_)
        }

        return self

    def transform(self, X):
        rows, cols, _ = extract_position_bins(
            X, bin_size=self.bin_size
        )

        selected_rows = []
        selected_cols = []

        for row_idx, feature in zip(rows, cols):
            col_idx = self.bin_to_idx_.get(feature)

            if col_idx is not None:
                selected_rows.append(row_idx)
                selected_cols.append(col_idx)

        data = np.ones(len(selected_rows), dtype=np.float32)

        return csr_matrix(
            (data, (selected_rows, selected_cols)),
            shape=(len(X), len(self.selected_bins_)),
            dtype=np.float32
        )# 피처 구성 E + Position-bin CV

from scipy.sparse import hstack

class FeaturesEWithPositionBins(BaseEstimator, TransformerMixin):
    def __init__(self, bin_size=50, min_patients=5):
        self.bin_size = bin_size
        self.min_patients = min_patients

    def fit(self, X, y=None):
        self.position_transformer_ = PositionBinTransformer(
            bin_size=self.bin_size,
            min_patients=self.min_patients
        )
        self.position_transformer_.fit(X)
        return self

    def transform(self, X):
        # 기존 피처 구성 E
        X_base = make_features_E_sparse(X)

        # Fold 학습 데이터에서 선택한 Position-bin
        X_position = self.position_transformer_.transform(X)

        return hstack(
            [X_base, X_position],
            format="csr",
            dtype=np.float32
        )


pipeline_E_position_LR = Pipeline([
    ("features", FeaturesEWithPositionBins(
        bin_size=50,
        min_patients=5
    )),
    ("model", LogisticRegression(
        C=0.1,
        class_weight="balanced",
        max_iter=3000,
        solver="lbfgs"
    ))
])



## 2. LightGBM g3 변환기: Hotspot×3, Position-bin, Exact-position

In [5]:
# 유전자별 Hotspot 아미노산 위치
HOTSPOTS = {
    "IDH1": ["R132"],
    "IDH2": ["R140", "R172"],
    "BRAF": ["V600"],
    "HRAS": ["Q61", "G13", "G12"],
    "KIT": ["D816", "N822", "K642"],
    "CTNNB1": ["S33", "S37", "S45", "T41", "G34", "D32"],
    "U2AF1": ["S34"],
    "PIK3CA": ["E545", "H1047", "E542"],
    "EGFR": ["L858", "G598", "A289"],
    "RET": ["M918"],
    "RAC1": ["P29"],
    "SPOP": ["F133", "W131", "F102", "Y87"],
    "EZH2": ["Y641", "Y646"],
    "TP53": ["R175", "R248", "R273"],
    "PTEN": ["R130"],
}


assert sum(map(len,HOTSPOTS.values())) == 35

In [6]:
def make_hotspot_features(X):
    hotspot_data = {}

    # 일반 Hotspot
    for gene, positions in HOTSPOTS.items():
        if gene not in X.columns:
            raise ValueError(f"유전자 컬럼 없음: {gene}")

        tokens = X[gene].fillna("WT").astype(str).str.split()

        for position in positions:
            col_name = f"hotspot_{gene}_{position}"

            hotspot_data[col_name] = tokens.map(
                lambda variants: int(
                    any(v.startswith(position) for v in variants)
                )
            ).astype(np.int8)

    # NPM1 frameshift
    if "NPM1" not in X.columns:
        raise ValueError("유전자 컬럼 없음: NPM1")

    hotspot_data["hotspot_NPM1_fs"] = (
        X["NPM1"]
        .fillna("WT")
        .astype(str)
        .str.contains("fs", case=False, regex=False)
        .astype(np.int8)
    )

    return pd.DataFrame(hotspot_data, index=X.index)




In [7]:
from lightgbm import LGBMClassifier
from sklearn.base import BaseEstimator, TransformerMixin
from collections import Counter

# 변이 토큰의 시작에 있는 단일 영문자 + 숫자만 인식: V600E -> V600, R132H -> R132.
# 468_469LG처럼 숫자로 시작하거나 복잡한 표기는 의도적으로 제외.
EXACT_POSITION_RE = re.compile(r'^([A-Za-z])(\d+)')

def patient_exact_positions(X):
    result = []
    for row in X.itertuples(index=False, name=None):
        found = set()
        for gene, value in zip(X.columns, row):
            if gene in ('ID', 'SUBCLASS') or pd.isna(value) or str(value).strip().upper() == 'WT':
                continue
            for token in str(value).split():
                match = EXACT_POSITION_RE.match(token)
                if match:
                    found.add(f'{gene}_{match.group(1).upper()}{match.group(2)}')
        result.append(found)
    return result

class ExactPositionTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, min_patients=5):
        self.min_patients = min_patients

    def fit(self, X, y=None):
        counts = Counter(key for keys in patient_exact_positions(X) for key in keys)
        self.selected_ = sorted(k for k,v in counts.items() if v >= self.min_patients)
        self.mapping_ = {key: i for i,key in enumerate(self.selected_)}
        return self

    def transform(self, X):
        if not hasattr(self, 'mapping_'):
            raise RuntimeError('ExactPositionTransformer.fit()을 먼저 실행하세요')
        rows, cols = [], []
        for i, keys in enumerate(patient_exact_positions(X)):
            for key in keys:
                j = self.mapping_.get(key)
                if j is not None:
                    rows.append(i); cols.append(j)
        return csr_matrix((np.ones(len(rows),dtype=np.float32),(rows,cols)),
                          shape=(len(X),len(self.selected_)),dtype=np.float32)

class FeaturesEWithExactPositions(BaseEstimator, TransformerMixin):
    def __init__(self, min_patients=5):
        self.min_patients = min_patients
    def fit(self, X, y=None):
        self.exact_ = ExactPositionTransformer(self.min_patients).fit(X)
        return self
    def transform(self, X):
        return hstack([make_features_E_sparse(X), self.exact_.transform(X)],
                      format='csr',dtype=np.float32)



In [8]:
class FeaturesEHotspot3BinExact(BaseEstimator, TransformerMixin):
    def __init__(self, bin_size=50, min_patients=5, hotspot_multiplier=3):
        self.bin_size=bin_size
        self.min_patients=min_patients
        self.hotspot_multiplier=hotspot_multiplier

    def fit(self, X, y=None):
        # 선택은 반드시 이 fit에 전달된 학습 Fold 데이터에서만 수행
        self.position_transformer_ = PositionBinTransformer(
            bin_size=self.bin_size, min_patients=self.min_patients).fit(X)
        self.exact_transformer_ = ExactPositionTransformer(
            min_patients=self.min_patients).fit(X)
        return self

    def transform(self, X):
        base = make_features_E_sparse(X)
        hotspot = csr_matrix(make_hotspot_features(X).to_numpy(dtype=np.float32)
                             * self.hotspot_multiplier)
        bins = self.position_transformer_.transform(X)
        exact = self.exact_transformer_.transform(X)
        return hstack([base, hotspot, bins, exact], format='csr', dtype=np.float32)

G3_PARAMS = dict(n_estimators=800, learning_rate=0.02, num_leaves=15,
                 min_child_samples=5, colsample_bytree=0.2, subsample=0.8,
                 subsample_freq=1, class_weight='balanced', random_state=42,
                 verbosity=-1, n_jobs=-1)
lgbm_g3 = Pipeline([
    ('features', FeaturesEHotspot3BinExact(bin_size=50, min_patients=5, hotspot_multiplier=3)),
    ('model', LGBMClassifier(**G3_PARAMS))
])
_demo = pd.DataFrame({'BRAF':['V600E','V600K','WT','V600E','V600E','V600E'], 'IDH1':['WT']*6})
assert ExactPositionTransformer(5).fit(_demo).selected_ == ['BRAF_V600']
print('g3 피처 변환기 준비 완료')

g3 피처 변환기 준비 완료


## 3. 신규 모델 2개 정의

기존 `FeaturesEWithPositionBins`, `FeaturesEHotspot3BinExact`, `make_hotspot_features`를 재사용합니다.
선택형 구간/정확 위치 피처는 각 학습 fold의 `fit()`에서만 결정합니다.


In [9]:
from scipy.sparse import hstack
from sklearn.pipeline import Pipeline

class FeaturesEHotspot3Bins(BaseEstimator, TransformerMixin):
    def __init__(self, bin_size=50, min_patients=5, hotspot_multiplier=3):
        self.bin_size=bin_size
        self.min_patients=min_patients
        self.hotspot_multiplier=hotspot_multiplier
    def fit(self, X, y=None):
        self.bins_=PositionBinTransformer(self.bin_size,self.min_patients).fit(X)
        return self
    def transform(self, X):
        e=make_features_E_sparse(X)
        hotspot=csr_matrix(make_hotspot_features(X).to_numpy(dtype=np.float32))
        bins=self.bins_.transform(X)
        return hstack([e,hotspot*self.hotspot_multiplier,bins],format='csr',dtype=np.float32)

lr_hotspot=Pipeline([
    ('features',FeaturesEHotspot3Bins(50,5,3)),
    ('model',LogisticRegression(C=0.1,class_weight='balanced',max_iter=3000,solver='lbfgs'))
])
K=len(le.classes_)
XGB_PARAMS=dict(
    n_estimators=250,max_depth=3,learning_rate=0.05,
    subsample=0.8,colsample_bytree=0.8,
    objective='multi:softprob',num_class=K,eval_metric='mlogloss',
    tree_method='hist',device='cpu',random_state=SEED,n_jobs=-1
)
xgb_e=XGBClassifier(**XGB_PARAMS)
xgb_position=Pipeline([
    ('features',FeaturesEHotspot3BinExact(bin_size=50,min_patients=5,hotspot_multiplier=3)),
    ('model',XGBClassifier(**XGB_PARAMS))
])
lgbm_g3=Pipeline([
    ('features',FeaturesEHotspot3BinExact(bin_size=50,min_patients=5,hotspot_multiplier=3)),
    ('model',LGBMClassifier(**G3_PARAMS))
])
lr_position=pipeline_E_position_LR

def invsqrt_weights(yy):
    yy=np.asarray(yy,dtype=int)
    counts=np.bincount(yy,minlength=K)
    if np.any(counts==0): raise ValueError('학습 fold에 누락된 클래스')
    weights=1/np.sqrt(counts[yy])
    return (weights/weights.mean()).astype(np.float32)

def checked_proba(p,n,where):
    p=np.asarray(p,dtype=np.float64)
    assert p.shape==(n,K),f'{where} shape={p.shape} expected={(n,K)}'
    assert np.isfinite(p).all() and np.allclose(p.sum(axis=1),1,atol=1e-4),where
    return p

def fit_model(model,X_train,y_train,weighted=False,pipeline=False):
    params=({'model__sample_weight':invsqrt_weights(y_train)} if pipeline else
            {'sample_weight':invsqrt_weights(y_train)}) if weighted else {}
    fitted=clone(model).fit(X_train,y_train,**params)
    assert np.array_equal(fitted.classes_,np.arange(K)), '클래스 순서 불일치'
    return fitted

def cv_predict(model,X,labels,cv_folds,weighted=False,pipeline=False):
    oof=np.zeros((len(labels),K),dtype=np.float64)
    seen=np.zeros(len(labels),dtype=np.int8)
    for i,(tr,va) in enumerate(cv_folds,1):
        m=fit_model(model,X.iloc[tr],np.asarray(labels)[tr],weighted,pipeline)
        oof[va]=checked_proba(m.predict_proba(X.iloc[va]),len(va),f'fold {i}')
        seen[va]+=1
        print(f'  fold {i}/{len(cv_folds)} 완료')
    assert np.all(seen==1), 'OOF coverage 실패'
    return checked_proba(oof,len(labels),'OOF')

def scores_by_fold(proba,labels,cv_folds):
    return np.array([
        f1_score(np.asarray(labels)[va],proba[va].argmax(1),average='macro',zero_division=0)
        for _,va in cv_folds
    ])
print('두 후보 모델 준비 완료 — XGB CPU')


두 후보 모델 준비 완료 — XGB CPU


## 4. 개발 CV: 기준(3개) vs A(4개) vs B(3개)

**개발 데이터만** 사용합니다. 기준 3개도 동일 CPU·동일 fold로 재산출하여 공정 비교합니다.


In [10]:
cv_folds=list(folds)
print('기준 XGB E')
p_x=cv_predict(xgb_e,X_dev_E,y_dev,cv_folds,weighted=True)
print('기준 LR position')
p_lr=cv_predict(lr_position,X_dev_raw,y_dev,cv_folds)
print('기준 LightGBM g3')
p_g=cv_predict(lgbm_g3,X_dev_raw,y_dev,cv_folds)
print('신규 LR hotspot')
p_hot=cv_predict(lr_hotspot,X_dev_raw,y_dev,cv_folds)
print('신규 XGB position (CPU)')
p_xpos=cv_predict(xgb_position,X_dev_raw,y_dev,cv_folds,weighted=True,pipeline=True)

cv_probs={
    'baseline':checked_proba((p_x+p_lr+p_g)/3,len(y_dev),'baseline'),
    'A':checked_proba((p_x+p_lr+p_g+p_hot)/4,len(y_dev),'A'),
    'B':checked_proba((p_xpos+p_lr+p_g)/3,len(y_dev),'B')
}
cv_scores={k:scores_by_fold(v,y_dev,cv_folds) for k,v in cv_probs.items()}
cv_delta={k:float(cv_scores[k].mean()-cv_scores['baseline'].mean()) for k in ('A','B')}
cv_eligible={k:cv_delta[k]>0 for k in ('A','B')}
comparison=pd.DataFrame([
    dict(candidate=k,cv_mean=float(s.mean()),cv_std=float(s.std()),
         delta_vs_baseline=0 if k=='baseline' else cv_delta[k],
         better_folds='—' if k=='baseline' else f"{int(np.sum(s>cv_scores['baseline']))}/5",
         holdout_eligible=(k!='baseline' and cv_eligible[k]))
    for k,s in cv_scores.items()
])
display(comparison)
print('개발 CV에서 개선된 후보:',[k for k in ('A','B') if cv_eligible[k]])


기준 XGB E
  fold 1/5 완료
  fold 2/5 완료
  fold 3/5 완료
  fold 4/5 완료
  fold 5/5 완료
기준 LR position
  fold 1/5 완료
  fold 2/5 완료
  fold 3/5 완료
  fold 4/5 완료
  fold 5/5 완료
기준 LightGBM g3
  fold 1/5 완료
  fold 2/5 완료
  fold 3/5 완료
  fold 4/5 완료
  fold 5/5 완료
신규 LR hotspot
  fold 1/5 완료
  fold 2/5 완료
  fold 3/5 완료
  fold 4/5 완료
  fold 5/5 완료
신규 XGB position (CPU)
  fold 1/5 완료
  fold 2/5 완료
  fold 3/5 완료
  fold 4/5 완료
  fold 5/5 완료


,candidate,cv_mean,cv_std,delta_vs_baseline,better_folds,holdout_eligible
0,baseline,0.512631,0.014326,0.000000,—,False
1,A,0.512468,0.012955,-0.000163,2/5,False
2,B,0.513529,0.008207,0.000898,3/5,True


개발 CV에서 개선된 후보: ['B']


## 5. 고정 홀드아웃: CV 통과 후보만 1회 평가

기준 모델은 통과 후보가 있을 때에만 한 번 학습·평가합니다. A/B는 **개발 CV가 기준보다 높을 때만** 홀드아웃에 진입합니다.


In [11]:
holdout_results={}
passed={}
if not any(cv_eligible.values()):
    print('A/B 모두 개발 CV 미개선 → 홀드아웃 생략, 저장·PR 보류')
else:
    assert len(dev_idx)==4960 and len(hold_idx)==1241
    row_hash=pd.util.hash_pandas_object(X_raw.fillna('<MISSING>').astype(str),index=False).to_numpy()
    assert not set(row_hash[dev_idx]) & set(row_hash[hold_idx]),'개발/홀드아웃 원본 중복'
    y_hold=np.asarray(y)[hold_idx]
    raw_tr=X_raw.iloc[dev_idx]; raw_hold=X_raw.iloc[hold_idx]
    e_tr=X_E.iloc[dev_idx]; e_hold=X_E.iloc[hold_idx]

    mx=fit_model(xgb_e,e_tr,y_dev,weighted=True)
    mlr=fit_model(lr_position,raw_tr,y_dev)
    mg=fit_model(lgbm_g3,raw_tr,y_dev)
    hx=checked_proba(mx.predict_proba(e_hold),len(y_hold),'holdout XGB')
    hlr=checked_proba(mlr.predict_proba(raw_hold),len(y_hold),'holdout LR')
    hg=checked_proba(mg.predict_proba(raw_hold),len(y_hold),'holdout g3')
    baseline_hold=(hx+hlr+hg)/3
    base_hold_score=float(f1_score(y_hold,baseline_hold.argmax(1),average='macro',zero_division=0))
    print('기준 홀드아웃 Macro F1:',base_hold_score)

    for key in ('A','B'):
        if not cv_eligible[key]:
            passed[key]=False
            print(key,'CV 미개선 → 홀드아웃 생략')
            continue
        if key=='A':
            mh=fit_model(lr_hotspot,raw_tr,y_dev)
            candidate=(hx+hlr+hg+checked_proba(mh.predict_proba(raw_hold),len(y_hold),'holdout hotspot'))/4
        else:
            mp=fit_model(xgb_position,raw_tr,y_dev,weighted=True,pipeline=True)
            candidate=(checked_proba(mp.predict_proba(raw_hold),len(y_hold),'holdout XGBpos')+hlr+hg)/3
        score=float(f1_score(y_hold,candidate.argmax(1),average='macro',zero_division=0))
        holdout_results[key]={'baseline':base_hold_score,'candidate':score,'delta':score-base_hold_score}
        passed[key]=bool(cv_eligible[key] and score >= base_hold_score - 0.02)  # 팀 규칙: CV 개선 + 홀드아웃 하락 0.02 이내
        print(key,'홀드아웃:',holdout_results[key],'통과:',passed[key])
    display(pd.DataFrame.from_dict(holdout_results,orient='index'))
print('최종 통과 후보:',[k for k in ('A','B') if passed.get(k,False)])
assert not passed.get('A', False), '후보 A는 이번 실행에서 제외해야 합니다'
assert passed.get('B', False), '후보 B가 팀 기준을 통과하지 않았습니다. 전체 OOF/Test를 진행하지 마세요.'


기준 홀드아웃 Macro F1: 0.5216563463792439
A CV 미개선 → 홀드아웃 생략
B 홀드아웃: {'baseline': 0.5216563463792439, 'candidate': 0.5133419796679508, 'delta': -0.008314366711293042} 통과: True


,baseline,candidate,delta
B,0.521656,0.513342,-0.008314


최종 통과 후보: ['B']


## 6. 후보 B 전용: 전체 OOF/Test 계산 (승인된 실험)

기존 예측 파일의 메타데이터 스키마·행 순서가 확실하지 않으므로 **기존 `.npy`를 전혀 불러오지 않고** B 구성 모델 3개를 동일한 전체 학습 데이터, 동일한 `get_folds`에서 다시 계산합니다. A는 실행하지 않습니다. 각 모델의 Test는 전체 train 재학습 결과입니다.

**실행:** 앞의 1~16 셀을 실행한 뒤 이 셀과 저장 셀을 실행하세요. 기존 실험 결과는 재사용하지 않으며 CPU 학습에 시간이 걸립니다.


In [12]:
import os, tempfile
PRED = ROOT / 'experiments' / 'preds'
PRED.mkdir(parents=True, exist_ok=True)
NEW_STEMS = {
    'xpos': 'A_model__xgb_position_hotspot3_bin_exact_11',
    'lr': 'A_model__lr_position_recomputed_for_B_11',
    'g3': 'A_model__lgbm_g3_recomputed_for_B_11',
    'B': 'A_model__ensemble_11_B_xgb_position_three_equal',
}
N, NT = len(y), len(X_test_raw)
generated = {}

def full_oof_test(model, raw, raw_test, folds_full, weighted=False, pipeline=False):
    oof = cv_predict(model, raw, y, folds_full, weighted, pipeline)
    fitted = fit_model(model, raw, y, weighted, pipeline)
    test_prob = checked_proba(fitted.predict_proba(raw_test), NT, 'full train Test')
    return oof, test_prob

if not RUN_FULL_CV:
    print('RUN_FULL_CV=False → 전체 OOF/Test 생략')
elif not passed.get('B', False):
    print('B가 팀 기준 미통과 → 전체 OOF/Test 생략')
else:
    full_folds = list(get_folds(y, X=X_raw, classes=le.classes_))
    assert len(full_folds) == 5
    print('B 구성 1/3: XGB 위치 (CPU, fold별 1/sqrt 빈도 가중치)')
    xpos_oof, xpos_test = full_oof_test(
        xgb_position, X_raw, X_test_raw, full_folds, weighted=True, pipeline=True)
    print('B 구성 2/3: 위치 LR')
    lr_oof, lr_test = full_oof_test(
        lr_position, X_raw, X_test_raw, full_folds)
    print('B 구성 3/3: LGBM g3')
    g_oof, g_test = full_oof_test(
        lgbm_g3, X_raw, X_test_raw, full_folds)
    b_oof = checked_proba((xpos_oof + lr_oof + g_oof) / 3, N, 'B OOF')
    b_test = checked_proba((xpos_test + lr_test + g_test) / 3, NT, 'B Test')
    generated = {
        'xpos': (xpos_oof, xpos_test),
        'lr': (lr_oof, lr_test),
        'g3': (g_oof, g_test),
        'B': (b_oof, b_test),
    }
    print('B 전체 OOF Macro F1:', f1_score(y, b_oof.argmax(1), average='macro', zero_division=0))
    print('B Test 확률 shape:', b_test.shape)


B 구성 1/3: XGB 위치 (CPU, fold별 1/sqrt 빈도 가중치)
  fold 1/5 완료
  fold 2/5 완료
  fold 3/5 완료
  fold 4/5 완료
  fold 5/5 완료
B 구성 2/3: 위치 LR
  fold 1/5 완료
  fold 2/5 완료
  fold 3/5 완료
  fold 4/5 완료
  fold 5/5 완료
B 구성 3/3: LGBM g3
  fold 1/5 완료
  fold 2/5 완료
  fold 3/5 완료
  fold 4/5 완료
  fold 5/5 완료
B 전체 OOF Macro F1: 0.5264138633361299
B Test 확률 shape: (2546, 26)


## 7. 후보 B 전체 OOF/Test 저장 (제출 CSV 없음)

신규 XGB 위치, 재계산한 위치 LR, 재계산한 LGBM g3 및 B 앙상블을 각각 OOF/Test/메타데이터로 저장합니다. 기존 파일을 임의로 덮어쓰지 않습니다.


In [13]:
def save_npy(stem,kind,arr):
    path=PRED/f'{stem}__{kind}.npy'
    arr=np.asarray(checked_proba(arr,len(arr),path.name),dtype=np.float32)
    if path.exists():
        existing=np.load(path,allow_pickle=False)
        if existing.shape!=arr.shape or not np.allclose(existing,arr,atol=1e-5):
            raise ValueError('기존 예측과 불일치, 덮어쓰기 금지: '+str(path))
        print('동일 파일 유지:',path.name);return
    fd,tmp=tempfile.mkstemp(dir=PRED,suffix='.npy');os.close(fd)
    try:
        np.save(tmp,arr);os.replace(tmp,path)
    finally:
        if os.path.exists(tmp):os.unlink(tmp)
    print('저장:',path.name)

def save_json(stem,details):
    path=PRED/f'{stem}__meta.json'
    obj={'classes':list(map(str,le.classes_)),'train_rows':N,'test_rows':NT,
         'folds':5,'seed':int(SEED),'test_fit':'full_train',**details}
    if path.exists():
        if json.loads(path.read_text(encoding='utf-8'))!=obj:
            raise ValueError('기존 메타와 불일치, 덮어쓰기 금지: '+str(path))
        print('동일 메타 유지:',path.name);return
    path.write_text(json.dumps(obj,ensure_ascii=False,indent=2),encoding='utf-8')
    print('저장:',path.name)

if not SAVE_ARTIFACTS:
    print('SAVE_ARTIFACTS=False → 저장 생략')
elif not RUN_FULL_CV or not generated:
    print('전체 예측 미생성 또는 통과 후보 없음 → 저장 생략')
else:
    details = {
        'xpos': {'model': 'weighted XGB E + hotspot x3 + bin + exact position',
                 'params': XGB_PARAMS, 'sample_weight': 'inverse_sqrt_frequency', 'device': 'cpu'},
        'lr': {'model': 'position-bin LR recomputed full OOF for B', 'source': 'lr_position'},
        'g3': {'model': 'LightGBM g3 recomputed full OOF for B', 'params': G3_PARAMS},
        'B': {'model': '3-model equal ensemble B',
              'sources': [NEW_STEMS['xpos'], NEW_STEMS['lr'], NEW_STEMS['g3']],
              'weights': [1/3]*3, 'dev_cv_delta': cv_delta['B'],
              'holdout_delta': holdout_results['B']['delta'],
              'validation_policy': 'CV improves and holdout drop <= 0.02'},
    }
    for key,(oof,tst) in generated.items():
        stem=NEW_STEMS[key]
        save_npy(stem,'oof',oof)
        save_npy(stem,'test',tst)
        save_json(stem,details[key])
    print('통과 후보 저장 완료 — 제출 CSV 생성 안 함')


저장: A_model__xgb_position_hotspot3_bin_exact_11__oof.npy
저장: A_model__xgb_position_hotspot3_bin_exact_11__test.npy
저장: A_model__xgb_position_hotspot3_bin_exact_11__meta.json
저장: A_model__lr_position_recomputed_for_B_11__oof.npy
저장: A_model__lr_position_recomputed_for_B_11__test.npy
저장: A_model__lr_position_recomputed_for_B_11__meta.json
저장: A_model__lgbm_g3_recomputed_for_B_11__oof.npy
저장: A_model__lgbm_g3_recomputed_for_B_11__test.npy
저장: A_model__lgbm_g3_recomputed_for_B_11__meta.json
저장: A_model__ensemble_11_B_xgb_position_three_equal__oof.npy
저장: A_model__ensemble_11_B_xgb_position_three_equal__test.npy
저장: A_model__ensemble_11_B_xgb_position_three_equal__meta.json
통과 후보 저장 완료 — 제출 CSV 생성 안 함


## 8. PR 전 확인

- `git fetch origin`, `git switch dev`, `git pull --ff-only origin dev` 후 새 기능 브랜치에서 작업하세요.
- 이 노트북과 **실제 통과하여 생성된 신규** `.npy`, `.json`만 커밋합니다.
- 기존 10번, 08/09번, `submit/` 및 `submissions/`는 변경하지 않습니다.
- 11번은 10번의 3모델 앙상블을 **기준으로 비교**하는 실험이며, 점수는 실제 실행 후 확정합니다.
- **홀드아웃 반복 튜닝 금지.** 같은 고정 홀드아웃을 이미 사용한 실험들이 있으므로 결과 해석에 유의하세요.
